In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

In [3]:
# 1. SETUP MODEL
MODEL_NAME = "indobenchmark/indobert-base-p2"
NUM_LABELS = 3  # rendah, sedang, tinggi

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# AutoModelForSequenceClassification = IndoBERT + Dense layer + Softmax
# Langsung siap untuk klasifikasi!
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
# 2. BUAT DATASET CLASS
class MentalHealthDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [5]:
# 3. CONTOH DATA
train_texts = [
    "Aku baik-baik saja, tidur cukup dan tidak stres",
    "Kadang merasa khawatir tapi masih bisa diatasi",
    "Aku sangat cemas, tidak bisa tidur, dan merasa putus asa",
    "Hidupku normal, pekerjaan lancar dan keluarga harmonis",
    "Sering merasa sedih tapi tidak tahu alasannya",
    "Tidak bisa berkonsentrasi, selalu merasa lelah dan hopeless"
]
train_labels = [0, 1, 2, 0, 1, 2]  # 0=rendah, 1=sedang, 2=tinggi

dataset = MentalHealthDataset(train_texts, train_labels, tokenizer)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

In [6]:
# 4. TRAINING LOOP
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

model.train()
for epoch in range(3):
    total_loss = 0
    for batch in dataloader:
        optimizer.zero_grad()

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(dataloader):.4f}")

Epoch 1 | Loss: 1.0608
Epoch 2 | Loss: 0.8475
Epoch 3 | Loss: 0.5293


In [7]:
# 5. INFERENCE
def predict_mental_health(teks):
    model.eval()
    label_map = {0: "Rendah", 1: "Sedang", 2: "Tinggi"}

    inputs = tokenizer(
        teks,
        max_length=128,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=1)
        predicted_class = torch.argmax(probabilities, dim=1).item()

    prob_list = probabilities[0].tolist()

    return {
        "prediksi": label_map[predicted_class],
        "probabilitas": {
            "Rendah": f"{prob_list[0]*100:.1f}%",
            "Sedang": f"{prob_list[1]*100:.1f}%",
            "Tinggi": f"{prob_list[2]*100:.1f}%"
        }
    }

# Test
hasil = predict_mental_health("Saya sangat stres dan tidak bisa tidur berhari-hari")
print(hasil)

{'prediksi': 'Tinggi', 'probabilitas': {'Rendah': '22.3%', 'Sedang': '30.5%', 'Tinggi': '47.2%'}}
